# Path Finding — SCIP Semantic Index

Demonstrates path finding algorithms applied to SCIP Semantic Index module and artifact dependency graphs.

**Key difference from TypeScript/Java path finding:**  
SCIP dependency graphs may contain cycles (circular module dependencies). The longest path algorithm requires a Directed Acyclic Graph (DAG). To handle cycles, Strongly Connected Components (SCC) are computed first — each cycle is condensed into a single SCC component node. Path finding then runs on the acyclic component graph (DAG).

**All Pairs Shortest Path (APSP)** works on the original graph (handles cycles naturally).  
**Longest Path** runs on the SCC component DAG.

### References
- [SCIP — Semantic Code Intelligence Protocol](https://github.com/sourcegraph/scip)
- [Neo4j GDS — All Pairs Shortest Path](https://neo4j.com/docs/graph-data-science/current/algorithms/all-pairs-shortest-path)
- [Neo4j GDS — Longest Path](https://neo4j.com/docs/graph-data-science/current/algorithms/dag/longest-path)
- [Neo4j GDS — Strongly Connected Components](https://neo4j.com/docs/graph-data-science/current/algorithms/strongly-connected-components)

In [ ]:
import os
import typing
import pandas as pd
import matplotlib.pyplot as plot
import numpy as np
from neo4j import GraphDatabase

In [ ]:
# Please set the environment variable "NEO4J_INITIAL_PASSWORD" in your shell 
# before starting jupyter notebook to provide the password for the user "neo4j".
# It is not recommended to hardcode the password into jupyter notebook for security reasons.

driver = GraphDatabase.driver(uri="bolt://localhost:7687", auth=("neo4j", os.environ.get("NEO4J_INITIAL_PASSWORD")))
driver.verify_connectivity()

In [ ]:
#The following cell uses the build-in %html "magic" to override the CSS style for tables to a much smaller size.
#This is especially needed for PDF export of tables with multiple columns.

In [ ]:
%%html
<style>
/* CSS style for smaller dataframe tables. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

In [ ]:
MAIN_COLOR_MAP = 'nipy_spectral'

pd.set_option('display.max_colwidth', 300)

## Helper functions

### Query helpers

In [ ]:
def get_cypher_query_from_file(filename: str) -> str:
    """Read and join all lines of a Cypher file into a single query string."""
    with open(filename) as file:
        return ' '.join(file.readlines())


def query_cypher_to_data_frame(
    filename: str,
    parameters: typing.Optional[typing.Dict[str, typing.Any]] = None,
) -> pd.DataFrame:
    """
    Execute the Cypher query from a file and return the result as a DataFrame.

    Args:
        filename: Path to the file containing the Cypher query.
        parameters: Optional dict of Cypher parameters.
    """
    records, summary, keys = driver.execute_query(
        get_cypher_query_from_file(filename),
        parameters_=parameters or {},
    )
    return pd.DataFrame([r.values() for r in records], columns=keys)

### Projection helpers

In [ ]:
def create_directed_unweighted_projection(parameters: dict) -> bool:
    """
    Create a directed unweighted in-memory GDS projection with filtered subgraph.

    Returns True when data exists for the given node label, False when the graph has no matching nodes.
    The projection name is derived from `parameters['dependencies_projection']`.
    Creates both the projection and a "-cleaned" subgraph (filtering out zero-degree nodes).
    """
    data_projectable = query_cypher_to_data_frame(
        "../../../cypher/Dependencies_Projection/Dependencies_0_Check_Projectable.cypher",
        parameters,
    )
    if data_projectable.empty:
        print(f"No data for node label '{parameters.get('dependencies_projection_node')}'.")
        return False
    
    # Prepare projection (fill in default values for missing properties)
    query_cypher_to_data_frame(
        "../../../cypher/Dependencies_Projection/Dependencies_0_Prepare_Projection.cypher",
        parameters,
    )
    
    # Delete existing projections and subgraphs
    query_cypher_to_data_frame(
        "../../../cypher/Dependencies_Projection/Dependencies_1_Delete_Projection.cypher",
        parameters,
    )
    query_cypher_to_data_frame(
        "../../../cypher/Dependencies_Projection/Dependencies_2_Delete_Subgraph.cypher",
        parameters,
    )
    
    # Create the projection
    query_cypher_to_data_frame(
        "../queries/path-finding/Path_Finding_1_Create_Projection.cypher",
        parameters,
    )
    
    # Create filtered subgraph without zero-degree nodes (generic approach)
    # This uses GDS degree centrality to identify nodes with at least one relationship
    projection_name = parameters.get("dependencies_projection")
    cleaned_name = projection_name + "-cleaned"
    
    filter_query = f"""
    CALL gds.graph.filter(
        '{cleaned_name}',
        '{projection_name}',
        'true',
        '*'
    )
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount
    """
    
    subgraph_result = driver.execute_query(filter_query)
    records, summary, keys = subgraph_result
    
    # Check if subgraph was created with relationships
    if records and len(records) > 0:
        result_dict = {k: v for k, v in zip(keys, records[0].values())}
        relationship_count = result_dict.get("relationshipCount", 0)
        if relationship_count > 0:
            return True
    
    print(f"No relationships in subgraph for projection '{projection_name}'.")
    return False

### SCC helpers

The following functions implement the cycle-handling pipeline used for SCIP path finding.
Each step is idempotent: it checks whether its output already exists before re-running.

In [ ]:
def ensure_scc_computed(parameters: dict) -> None:
    """
    Detect and write Strongly Connected Component IDs to member nodes (idempotent).

    Skips SCC detection when communityStronglyConnectedComponentId is already set.
    Always re-creates StronglyConnectedComponent nodes and their DEPENDS_ON edges
    using MERGE, so repeated calls are safe.
    """
    scc_exists = query_cypher_to_data_frame(
        "../queries/strongly-connected-components/SCC_Exists.cypher", parameters
    )
    if scc_exists.empty:
        print("Writing SCC component IDs to nodes...")
        query_cypher_to_data_frame(
            "../queries/strongly-connected-components/SCC_Write.cypher", parameters
        )
    else:
        print("SCC component IDs already present, skipping detection.")

    query_cypher_to_data_frame(
        "../queries/strongly-connected-components/SCC_CreateNode.cypher", parameters
    )
    query_cypher_to_data_frame(
        "../queries/strongly-connected-components/SCC_CreateDependency.cypher", parameters
    )


def recreate_scc_components_projection(parameters: dict) -> None:
    """
    Drop and recreate the ephemeral in-memory -components GDS projection.

    The -components projection is not persisted across Neo4j restarts. It is always
    recreated here so downstream algorithms have a fresh, consistent view.
    Prerequisites: ensure_scc_computed() must have run at least once.
    """
    query_cypher_to_data_frame(
        "../queries/strongly-connected-components/SCC_TopologicalSort_Delete_Projection.cypher",
        parameters,
    )
    query_cypher_to_data_frame(
        "../queries/strongly-connected-components/SCC_TopologicalSort_Projection.cypher",
        parameters,
    )


def ensure_scc_topology_sorted(parameters: dict) -> None:
    """
    Compute topological sort on the SCC component DAG and propagate levels to member nodes (idempotent).

    Skips writing when topologicalSortMaxDistanceFromSource is already set on component nodes.
    Topological sort values are required for level labels in GraphViz visualizations and
    for the build level groupings in the longest path charts.
    Prerequisites: recreate_scc_components_projection() must have run in this session.
    """
    topology_exists = query_cypher_to_data_frame(
        "../queries/strongly-connected-components/SCC_TopologicalSort_Exists.cypher",
        parameters,
    )
    if topology_exists.empty:
        print("Writing topological sort to SCC components...")
        query_cypher_to_data_frame(
            "../queries/strongly-connected-components/SCC_TopologicalSort_Write.cypher",
            parameters,
        )
    else:
        print("Topological sort already present on SCC components, skipping.")

    query_cypher_to_data_frame(
        "../queries/strongly-connected-components/SCC_TopologicalSort_Propagate.cypher",
        parameters,
    )


def setup_scc_for_longest_path(parameters: dict) -> None:
    """
    Full SCC pipeline required before running the SCC-based longest path algorithm.

    Runs in order:
    1. Detect and write SCC component IDs (idempotent).
    2. Recreate the in-memory -components projection (always fresh).
    3. Compute topological sort on the component DAG (idempotent).

    Args:
        parameters: Dict with keys dependencies_projection, dependencies_projection_node,
                    dependencies_projection_weight_property.
    """
    ensure_scc_computed(parameters)
    recreate_scc_components_projection(parameters)
    ensure_scc_topology_sorted(parameters)

### Distribution analysis helpers

In [ ]:
def get_total_distance_distribution(data_frame: pd.DataFrame) -> pd.DataFrame:
    """Aggregate path count per distance across all projects."""
    return (
        data_frame
        .groupby('distance')[['pairCount', 'distanceTotalPairCount']]
        .agg({'pairCount': 'sum', 'distanceTotalPairCount': 'max'})
        .reset_index()
        .sort_values('distance')
    )


def get_max_distance_per_project(data_frame: pd.DataFrame) -> pd.DataFrame:
    """Return the maximum distance (diameter) per source project, sorted descending."""
    return (
        data_frame
        .groupby('sourceProject')['distance']
        .max()
        .reset_index()
        .rename(columns={'distance': 'maxDistance'})
        .sort_values('maxDistance', ascending=False)
    )


def pivot_distribution_by_project(data_frame: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot the per-project distribution into wide format (rows=distance, columns=project).

    Only includes intra-project pairs (isDifferentTargetProject == False).
    """
    intra = data_frame[data_frame['isDifferentTargetProject'] == False].copy()
    if intra.empty:
        return pd.DataFrame()
    return intra.pivot_table(
        index='distance',
        columns='sourceProject',
        values='pairCount',
        aggfunc='sum',
        fill_value=0,
    ).sort_index()

### Chart helpers

In [ ]:
def plot_distance_bar(data_frame: pd.DataFrame, title: str, ylabel: str = 'Number of paths') -> None:
    """Bar chart of path count per distance."""
    if data_frame.empty:
        print(f"No data for: {title}")
        return
    figure, axis = plot.subplots(figsize=(10, 5))
    data_frame.plot(kind='bar', x='distance', y='pairCount', ax=axis,
                    legend=False, grid=True, fontsize=8, cmap=MAIN_COLOR_MAP,
                    xlabel='Distance (number of hops)', ylabel=ylabel, title=title)
    axis.tick_params(axis='x', labelrotation=45)
    figure.tight_layout()
    plot.show()


def plot_distance_pie(data_frame: pd.DataFrame, title: str) -> None:
    """Pie chart of path count by distance."""
    if data_frame.empty:
        print(f"No data for: {title}")
        return
    total = data_frame['pairCount'].sum()

    def autopct(pct: float) -> str:
        return f'{pct:1.2f}% ({int(round(total * pct / 100))})'

    figure, axis = plot.subplots(figsize=(8, 8))
    data_frame.plot(kind='pie', y='pairCount', labels=data_frame['distance'],
                    labeldistance=None, legend=True, autopct=autopct,
                    explode=np.full(len(data_frame), 0.01),
                    textprops={'fontsize': 8}, pctdistance=1.2,
                    cmap=MAIN_COLOR_MAP, ax=axis, title=title)
    axis.set_ylabel('')
    axis.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='distance')
    figure.tight_layout()
    plot.show()


def plot_diameter_bar(data_frame: pd.DataFrame, title: str) -> None:
    """Bar chart of max distance (diameter) per project, descending."""
    if data_frame.empty:
        print(f"No data for: {title}")
        return
    top = data_frame.head(20)
    figure, axis = plot.subplots(figsize=(max(10, len(top) * 0.8 + 2), 5))
    top.plot(kind='bar', x='sourceProject', y='maxDistance', ax=axis,
             legend=False, grid=True, fontsize=8, cmap=MAIN_COLOR_MAP,
             xlabel='Project', ylabel='Max distance', title=title)
    axis.tick_params(axis='x', labelrotation=45)
    figure.tight_layout()
    plot.show()

## 1 — SCIP Module Path Finding

### 1.1 All Pairs Shortest Path (APSP)

Operates on the full SCIP module dependency graph. Cycles are handled naturally by the BFS-based algorithm.

In [ ]:
scip_module_parameters = {
    "dependencies_projection": "scip-module-path-finding-notebook",
    "dependencies_projection_node": "SemanticCodeIndexModule",
    "dependencies_projection_weight_property": "referenceCount",
    "dependencies_projection_language": "SCIP_Semantic_Index",
}

In [ ]:
is_scip_module_data_available = create_directed_unweighted_projection(scip_module_parameters)
print(f"SCIP Module projection available: {is_scip_module_data_available}")

In [ ]:
if is_scip_module_data_available:
    scip_module_apsp = query_cypher_to_data_frame(
        "../queries/path-finding/Path_Finding_5_All_pairs_shortest_path_distribution_per_project.cypher",
        scip_module_parameters,
    )
else:
    scip_module_apsp = pd.DataFrame()

print(f"APSP rows: {len(scip_module_apsp)}")

#### 1.1.1 APSP total distribution

In [ ]:
scip_module_apsp_total = get_total_distance_distribution(scip_module_apsp)
scip_module_apsp_total.head(30)

In [ ]:
plot_distance_bar(scip_module_apsp_total, 'SCIP Module — All Pairs Shortest Path Distribution')

In [ ]:
plot_distance_pie(scip_module_apsp_total, 'SCIP Module — All Pairs Shortest Path by Distance')

#### 1.1.2 APSP graph diameter per project

The graph diameter is the longest shortest path. Higher = deeper transitive dependency chains.

In [ ]:
scip_module_apsp_diameter = get_max_distance_per_project(scip_module_apsp)
scip_module_apsp_diameter.head(20)

In [ ]:
plot_diameter_bar(scip_module_apsp_diameter, 'SCIP Module — Graph Diameter per Project')

### 1.2 SCC-based Longest Path

Because the SCIP module graph may contain cycles, the Longest Path algorithm cannot run directly on it (it requires a DAG). The cycle-handling pipeline:

1. **Detect SCCs**: Find Strongly Connected Components. Each cycle becomes one component.
2. **Build component graph**: Create `StronglyConnectedComponent` nodes and `DEPENDS_ON` edges.
3. **Recreate -components projection**: The GDS in-memory projection is ephemeral — recreated every session.
4. **Topological sort**: Assign build levels to each component node.
5. **Longest path**: Run on the condensed DAG. Each cycle contributes 1 level.

In [ ]:
# SCC longest path uses a separate projection name to avoid conflicts with the APSP projection above
scip_module_topology_parameters = {
    "dependencies_projection": "scip-module-topology-notebook",
    "dependencies_projection_node": "SemanticCodeIndexModule",
    "dependencies_projection_weight_property": "referenceCount",
    "dependencies_projection_language": "SCIP_Semantic_Index",
}

In [ ]:
is_scip_module_topology_available = create_directed_unweighted_projection(scip_module_topology_parameters)
print(f"SCIP Module topology projection available: {is_scip_module_topology_available}")

In [ ]:
if is_scip_module_topology_available:
    setup_scc_for_longest_path(scip_module_topology_parameters)
    print("SCC pipeline complete.")

In [ ]:
if is_scip_module_topology_available:
    scip_module_longest = query_cypher_to_data_frame(
        "../queries/path-finding/SCC_Longest_paths_distribution_per_project.cypher",
        scip_module_topology_parameters,
    )
else:
    scip_module_longest = pd.DataFrame()

print(f"SCC Longest Path rows: {len(scip_module_longest)}")

#### 1.2.1 SCC Longest Path total distribution

In [ ]:
scip_module_longest_total = get_total_distance_distribution(scip_module_longest)
scip_module_longest_total.head(30)

In [ ]:
plot_distance_bar(scip_module_longest_total, 'SCIP Module — SCC Longest Path Distribution')

In [ ]:
plot_distance_pie(scip_module_longest_total, 'SCIP Module — SCC Longest Path by Distance')

#### 1.2.2 SCC Longest Path — max per project

In [ ]:
scip_module_longest_diameter = get_max_distance_per_project(scip_module_longest)
scip_module_longest_diameter.head(20)

In [ ]:
plot_diameter_bar(scip_module_longest_diameter, 'SCIP Module — Max SCC Longest Path per Project')

## 2 — SCIP Artifact Path Finding

Same pipeline as SCIP modules, but at the package/artifact level.

In [ ]:
scip_artifact_parameters = {
    "dependencies_projection": "scip-artifact-path-finding-notebook",
    "dependencies_projection_node": "SemanticCodeIndexArtifact",
    "dependencies_projection_weight_property": "referenceCount",
    "dependencies_projection_language": "SCIP_Semantic_Index",
}

In [ ]:
is_scip_artifact_data_available = create_directed_unweighted_projection(scip_artifact_parameters)
print(f"SCIP Artifact projection available: {is_scip_artifact_data_available}")

In [ ]:
if is_scip_artifact_data_available:
    scip_artifact_apsp = query_cypher_to_data_frame(
        "../queries/path-finding/Path_Finding_5_All_pairs_shortest_path_distribution_per_project.cypher",
        scip_artifact_parameters,
    )
else:
    scip_artifact_apsp = pd.DataFrame()

print(f"SCIP Artifact APSP rows: {len(scip_artifact_apsp)}")

In [ ]:
scip_artifact_apsp_total = get_total_distance_distribution(scip_artifact_apsp)
scip_artifact_apsp_total.head(30)

In [ ]:
plot_distance_bar(scip_artifact_apsp_total, 'SCIP Artifact — All Pairs Shortest Path Distribution')

In [ ]:
scip_artifact_apsp_diameter = get_max_distance_per_project(scip_artifact_apsp)
plot_diameter_bar(scip_artifact_apsp_diameter, 'SCIP Artifact — Graph Diameter per Project')

In [ ]:
scip_artifact_topology_parameters = {
    "dependencies_projection": "scip-artifact-topology-notebook",
    "dependencies_projection_node": "SemanticCodeIndexArtifact",
    "dependencies_projection_weight_property": "referenceCount",
    "dependencies_projection_language": "SCIP_Semantic_Index",
}

is_scip_artifact_topology_available = create_directed_unweighted_projection(scip_artifact_topology_parameters)

if is_scip_artifact_topology_available:
    setup_scc_for_longest_path(scip_artifact_topology_parameters)
    scip_artifact_longest = query_cypher_to_data_frame(
        "../queries/path-finding/SCC_Longest_paths_distribution_per_project.cypher",
        scip_artifact_topology_parameters,
    )
else:
    scip_artifact_longest = pd.DataFrame()

print(f"SCIP Artifact SCC Longest Path rows: {len(scip_artifact_longest)}")

In [ ]:
scip_artifact_longest_total = get_total_distance_distribution(scip_artifact_longest)
plot_distance_bar(scip_artifact_longest_total, 'SCIP Artifact — SCC Longest Path Distribution')